# Chuard — calidad, limpieza y EDA de entrenamiento

El CSV original permanece inmutable. La inspección estructural describe el archivo
completo; las conclusiones y gráficos de desarrollo utilizan **solo entrenamiento**.
No se entrena ningún modelo ni se calculan métricas predictivas en este notebook.

La entrega incluye el manifiesto definitivo cuya huella coincide con el ganador.
La ruta alternativa de propuesta solo sirve para desarrollo cuando ese manifiesto
todavía no existe; no sustituye una asignación congelada ni acredita su aprobación.
La fuente del dataset, descarga, huella y guía de ejecución están en el README.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Abra el notebook desde la raíz de Chuard o notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import json

import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import DATA_REPORT_DIR, RANDOM_STATE, RAW_DATA_PATH, SPLIT_MANIFEST_PATH
from src.data import (
    audit_dataset,
    dataset_sha256,
    generate_eda_report,
    read_customer_csv,
    summarize_training,
    training_subset_from_manifest,
)
from src.features import build_features
from src.schema import DERIVED_COLUMNS
from src.train import split_positions

if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError("Obtenga el CSV original con python -m src.data download.")
original_hash = dataset_sha256(RAW_DATA_PATH)
quality = audit_dataset(RAW_DATA_PATH)
display(
    pd.Series(
        {
            "registros": quality["rows"],
            "columnas": quality["column_count"],
            "filas_duplicadas": quality["duplicate_rows"],
            "IDs_duplicados": quality["duplicate_customer_ids"],
            "TotalCharges_vacíos": quality["empty_values"]["TotalCharges"],
            "normalizaciones_a_cero": quality["total_charges_zero_normalizations"],
        }
    )
)
display(pd.Series(quality["target_counts"], name="Distribución estructural de Churn"))
print("SHA-256:", original_hash)
print("Antigüedad de los TotalCharges vacíos:", quality["total_charges_empty_tenure"])
print("No se eliminan clientes distintos que compartan perfil.")

In [ ]:
DATA_REPORT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = SPLIT_MANIFEST_PATH
if not manifest_path.is_file():
    manifest_path = DATA_REPORT_DIR / "splits_propuesta.json"
    if not manifest_path.is_file():
        normalized = read_customer_csv(RAW_DATA_PATH, mode="training").data
        labels = normalized["Churn"].map({"No": 0, "Yes": 1})
        proposal = {
            "dataset_sha256": original_hash,
            "random_state": RANDOM_STATE,
            "index_kind": "posición normalizada desde 0",
            "status": "propuesta_pendiente_confirmación_frente2",
            "splits": split_positions(labels),
        }
        manifest_path.write_text(json.dumps(proposal, indent=2), encoding="utf-8")
training = training_subset_from_manifest(RAW_DATA_PATH, manifest_path)
print("Manifiesto:", manifest_path.relative_to(root))
print("Estado:", training.attrs["split_status"])
print("Registros de entrenamiento:", len(training))
print("No se calculan conclusiones sobre validación/prueba.")

In [ ]:
before = training.copy(deep=True)
with_features = build_features(training)
pd.testing.assert_frame_equal(training, before)
assert "customerID" not in with_features and "Churn" not in with_features
display(with_features.loc[:, list(DERIVED_COLUMNS)].describe())
summary = summarize_training(training)
display(pd.DataFrame(summary["numeric"]).T)
print("Los límites IQR son diagnósticos: no eliminan ni truncan valores.")
print("Cargos altos propuestos (Q3 de entrenamiento):", summary["high_monthly_charges"])

In [ ]:
report = generate_eda_report(RAW_DATA_PATH, manifest_path, DATA_REPORT_DIR)
for item in report["conclusions"]:
    display(Image(filename=str(DATA_REPORT_DIR / item["figure"])))
    display(Markdown(item["conclusion"]))
assert dataset_sha256(RAW_DATA_PATH) == original_hash
print("CSV original intacto. Figuras:", len(report["figures"]))

## Entrega y límites

Los JSON/CSV/PNG de `reports/data/` registran huella, calidad, estadísticas, tamaños de
segmento y conclusiones observadas. Los importes usan unidades monetarias: el CSV no
documenta una moneda. Las asociaciones no prueban causas.

El mismo CSV y manifiesto congelado se reutilizan en modelos y explicabilidad. Las
medianas, codificación y escalado se ajustan exclusivamente dentro del pipeline y de
sus folds. La comparación de modelos y métricas se presenta en el notebook 02.
Este notebook y su copia ejecutada forman parte de la entrega académica en GitHub.